# Coleta de Dados

**Tema:** reviews de jogos coletadas na Steam Web API.

## Fontes de dados — decisão

| Fonte | Uso | Motivo |
|---|---|---|
| `store.steampowered.com/appreviews/<appid>` | **Fonte primária** (corpus) | Pública, sem chave, JSON, paginação por cursor. Traz o texto da review e metadados (recomendação, horas jogadas no momento da review, data, idioma). |
| `store.steampowered.com/api/appdetails` | Base complementar | Metadados do jogo (nome, gêneros, descrição, preço, data de lançamento) — necessários para as tarefas de recomendação e agentes. |


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "src"))

import json
import pandas as pd
from steam_api import collect_reviews, fetch_appdetails, hash_steam_id, USER_AGENT

pd.set_option("display.max_colwidth", 80)
print("User-Agent identificado para a coleta:", USER_AGENT)


User-Agent identificado para a coleta: FURB-PLN-Projeto-Etapa1/1.0 (uso academico; contato: 00jonathan.ribeiro@gmail.com)


## Jogos-alvo

Para atender ao critério de **abrangência e variedade** (peso 0,4 na dimensão "Base de
dados textuais"), os jogos foram escolhidos para cobrir gêneros, modos de jogo e
perfis de recepção diferentes — não apenas um único tipo de jogo:

| appid | Jogo | Por que entrou na amostra |
|---|---|---|
| 730 | Counter-Strike 2 | Altíssimo volume, comunidade PT-BR muito ativa, multiplayer competitivo |
| 413150 | Stardew Valley | Indie, single-player, tom calmo — contraste de vocabulário e sentimento |
| 252950 | Rocket League | Multiplayer casual/competitivo baseado em física, comunidade distinta da de CS2 |
| 945360 | Among Us | Casual multiplayer, reviews tipicamente muito curtas |
| 1426210 | It Takes Two | Cooperativo narrativo (só se joga em dupla), aclamado pela crítica — quase sem review negativa |


In [2]:
TARGET_GAMES = {
    730: "Counter-Strike 2",
    413150: "Stardew Valley",
    252950: "Rocket League",
    945360: "Among Us",
    1426210: "It Takes Two",
}
TARGET_GAMES


{730: 'Counter-Strike 2',
 413150: 'Stardew Valley',
 252950: 'Rocket League',
 945360: 'Among Us',
 1426210: 'It Takes Two'}

## Coleta de metadados dos jogos (`appdetails`)

Resposta bruta preservada em `data/raw/appdetails/<appid>.json` (URL, parâmetros e
instante da coleta), conforme a prática de proveniência da Aula 4.


In [3]:
games_rows = []
for appid, expected_name in TARGET_GAMES.items():
    details = fetch_appdetails(appid)
    if details is None:
        print(f"[aviso] appdetails falhou para {appid} ({expected_name})")
        continue
    games_rows.append({
        "appid": appid,
        "name": details.get("name"),
        "short_description": details.get("short_description"),
        "genres": ", ".join(g["description"] for g in details.get("genres", [])),
        "categories": ", ".join(c["description"] for c in details.get("categories", [])),
        "release_date": details.get("release_date", {}).get("date"),
        "developers": ", ".join(details.get("developers", [])),
        "publishers": ", ".join(details.get("publishers", [])),
        "is_free": details.get("is_free"),
        "price_brl": (details.get("price_overview") or {}).get("final_formatted"),
        "metacritic_score": (details.get("metacritic") or {}).get("score"),
    })

games_df = pd.DataFrame(games_rows)
games_df.to_csv("../data/processed/games.csv", index=False)
games_df


,appid,name,short_description,genres,categories,release_date,developers,publishers,is_free,price_brl,metacritic_score
0,730,Counter-Strike 2,"Ao longo de mais de duas décadas, o Counter‑Strike ofereceu uma experiência ...","Ação, Grátis para Jogar","Multijogador, Multijogador interplataformas, Cartas Colecionáveis Steam, Ste...",21 ago. 2012,Valve,Valve,True,NaN,NaN
1,413150,Stardew Valley,You've inherited your grandfather's old farm plot in Stardew Valley. Armed w...,"Indie, RPG, Simulação","Um jogador, Multijogador, Co-op, Co-op online, Co-op em LAN, Co-op, ecrã par...",26 fev. 2016,ConcernedApe,ConcernedApe,False,"R$ 24,99",89.0
2,252950,Rocket League®,Futebol e condução encontram-se uma vez mais na aguardada sequela do clássic...,"Ação, Indie, Corridas, Desporto","Um jogador, Multijogador, PvP, PvP online, PvP, ecrã partilhado/dividido, Co...",6 jul. 2015,Psyonix LLC,Psyonix LLC,False,NaN,86.0
3,945360,Among Us,Um jogo de grupo local ou online para 4 a 15 jogadores onde o futuro é ditad...,Casual,"Multijogador, PvP, PvP online, PvP em LAN, Co-op, Co-op online, Co-op em LAN...",16 nov. 2018,Innersloth,Innersloth,False,"R$ 14,99",85.0
4,1426210,It Takes Two,Entra na caminhada mais louca da tua vida em It Takes Two. Convida um amigo ...,"Ação, Aventura","Multijogador, Co-op, Co-op online, Co-op, ecrã partilhado/dividido, Ecrã par...",25 mar. 2021,Hazelight Studios,Electronic Arts,False,"R$ 59,70",89.0


## Coleta de reviews (`appreviews`)

Parâmetros usados:
- `filter=recent` — paginação estável por data de criação.
- `language=brazilian,english` — a Steam aceita lista de idiomas separada por
  vírgula neste endpoint (comportamento não documentado oficialmente, confirmado em
  teste manual antes de usar aqui). Restringe a coleta a português (Brasil) e inglês.
- `review_type=all`, `purchase_type=all` — sem restringir positivas/negativas ou forma
  de compra.
- `num_per_page=100` — máximo permitido por página.

Cada página crua é gravada em `data/raw/reviews/<appid>/page_XXX.json` antes de
qualquer transformação. Entre requisições há uma pausa (`REQUEST_DELAY_SECONDS`) e
reação a `429`/`503` com backoff, para manter uma carga proporcional (Aula 4).


In [4]:
MAX_REVIEWS_PER_GAME = 400
COLLECTION_LANGUAGE = "brazilian,english"

all_reviews = []
for appid, name in TARGET_GAMES.items():
    reviews = collect_reviews(appid, max_reviews=MAX_REVIEWS_PER_GAME, language=COLLECTION_LANGUAGE)
    print(f"{name} ({appid}): {len(reviews)} reviews coletadas")
    all_reviews.extend(reviews)

print(f"\nTotal geral: {len(all_reviews)} reviews")


Counter-Strike 2 (730): 400 reviews coletadas


Stardew Valley (413150): 400 reviews coletadas


Rocket League (252950): 400 reviews coletadas


Among Us (945360): 400 reviews coletadas


It Takes Two (1426210): 400 reviews coletadas

Total geral: 2000 reviews


## Consolidação em tabela

A partir daqui aplicamos apenas **achatamento de estrutura e tipagem** (extrair campos
do JSON aninhado, converter timestamps e strings numéricas) — não é limpeza ou
pré-processamento linguístico, que ficam para a próxima etapa do pipeline.

Campos pessoais do autor (`personaname`, `profile_url`, `avatar`, `persona_status`) são
descartados por minimização de dados (LGPD); o `steamid` é substituído por um hash,
preservando apenas a capacidade de identificar reviews do mesmo autor.


In [5]:
def flatten_review(review: dict) -> dict:
    author = review.get("author", {})
    return {
        "appid": review["appid"],
        "game_name": TARGET_GAMES.get(review["appid"]),
        "recommendationid": review.get("recommendationid"),
        "review": review.get("review"),
        "language": review.get("language"),
        "voted_up": review.get("voted_up"),
        "timestamp_created": pd.to_datetime(review.get("timestamp_created"), unit="s", utc=True),
        "timestamp_updated": pd.to_datetime(review.get("timestamp_updated"), unit="s", utc=True),
        "votes_up": review.get("votes_up"),
        "votes_funny": review.get("votes_funny"),
        "weighted_vote_score": float(review.get("weighted_vote_score", 0) or 0),
        "comment_count": review.get("comment_count"),
        "steam_purchase": review.get("steam_purchase"),
        "received_for_free": review.get("received_for_free"),
        "refunded": review.get("refunded"),
        "written_during_early_access": review.get("written_during_early_access"),
        "primarily_steam_deck": review.get("primarily_steam_deck"),
        "author_id_hash": hash_steam_id(author["steamid"]) if author.get("steamid") else None,
        "author_num_games_owned": author.get("num_games_owned"),
        "author_num_reviews": author.get("num_reviews"),
        "author_playtime_at_review_hours": round((author.get("playtime_at_review") or 0) / 60, 1),
        "author_playtime_forever_hours": round((author.get("playtime_forever") or 0) / 60, 1),
    }

reviews_df = pd.DataFrame(flatten_review(r) for r in all_reviews)
reviews_df.to_csv("../data/processed/reviews_raw.csv", index=False)
reviews_df.shape


(2000, 22)

In [6]:
reviews_df.head(3)


,appid,game_name,recommendationid,review,language,voted_up,timestamp_created,timestamp_updated,votes_up,votes_funny,...,steam_purchase,received_for_free,refunded,written_during_early_access,primarily_steam_deck,author_id_hash,author_num_games_owned,author_num_reviews,author_playtime_at_review_hours,author_playtime_forever_hours
0,730,Counter-Strike 2,235720753,would be pro if it werent laggy asf. stupid jitter ms,english,True,2026-09-20 20:52:51+00:00,2026-09-20 20:52:51+00:00,0,0,...,True,False,False,False,False,6b66c7288971adec,17,10,74.3,74.3
1,730,Counter-Strike 2,235720638,w game im gay and from israel,english,True,2026-09-20 20:51:03+00:00,2026-09-20 20:51:03+00:00,0,0,...,True,False,False,False,False,cb0288be78bc9e09,0,1,4.4,4.4
2,730,Counter-Strike 2,235720566,this is valoront for big boys,english,True,2026-09-20 20:49:46+00:00,2026-09-20 20:49:46+00:00,0,0,...,True,False,False,False,False,d7b2b3f68fc9279e,0,4,93.8,93.8


## Conferência da base coletada

Checagem de volume, abrangência (jogos) e variedade (idiomas, polaridade) — critérios
da dimensão "Base de dados textuais".


In [7]:
print("Total de reviews:", len(reviews_df))
print("\nReviews por jogo:")
print(reviews_df["game_name"].value_counts())
print("\nIdiomas declarados:")
print(reviews_df["language"].value_counts())
print("\nDistribuição voted_up (recomendação):")
print(reviews_df["voted_up"].value_counts(normalize=True).round(2))
print("\nTamanho do texto (caracteres) — describe:")
print(reviews_df["review"].fillna("").str.len().describe())


Total de reviews: 2000

Reviews por jogo:
game_name
Counter-Strike 2    400
Stardew Valley      400
Rocket League       400
Among Us            400
It Takes Two        400
Name: count, dtype: int64

Idiomas declarados:
language
english      1705
brazilian     295
Name: count, dtype: int64

Distribuição voted_up (recomendação):
voted_up
True     0.87
False    0.13
Name: proportion, dtype: float64

Tamanho do texto (caracteres) — describe:
count    2000.000000
mean      104.617000
std       344.715748
min         0.000000
25%         9.000000
50%        26.000000
75%        79.000000
max      8000.000000
Name: review, dtype: float64
